# Weight cap experiment: x_max in {1.0, 0.6, 0.3, 0.2}

How a per-asset weight cap changes performance.

- Training and merging are handled by `run_xmax_sweep.py`, which finishes one
  x_max completely before starting the next.
- Checkpoints carry an `_xm` tag, which keeps them separate from the uncapped
  (x_max = 1.0) results.
- The analysis reuses the existing modules; this notebook only calls them.

| Example filename | Meaning |
| --- | --- |
| `dfl_mdd_10_inds_h126_d20_l0.5_CLARABEL.pkl` | x_max = 1.0 (uncapped) |
| `dfl_mdd_10_inds_h126_xm0.3_d20_l0.5_CLARABEL.pkl` | x_max = 0.3 |


## 1. Setup

In [ ]:
import os, sys, pickle, subprocess, time, importlib, re
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

import benchmarks, performance, carryforward
for _m in (benchmarks, performance, carryforward):
    importlib.reload(_m)

from benchmarks import (build_bench_store, attach_date_idx,
                        compute_turnover, rebalance_dates)
from carryforward import apply_carryforward, parse_lb, parse_n1
from performance import compute_performance, apply_tc, build_equity_curve

# -- experiment settings --
N_STOCKS      = 10
HORIZON       = 126
REBAL         = 21
SOLVER        = "CLARABEL"
DELTA         = 20
LAM_LIST      = [0.3, 0.5, 0.7, 1.0]
LOOKBACK_LIST = [252, 504]
N1_LIST       = [0.1, 0.2, 0.3, 0.4]
XMAX_LIST     = [0.2, 0.3, 0.6]           # caps run in this experiment
XMAX_ALL      = XMAX_LIST + [1.0]    # plus the uncapped baseline

VAL_YEARS, TEST_YEARS, N_FOLDS = 5, 1, 8
CKPT_DIR   = "./checkpoint"
PLOT_DIR   = "./plots/xmax"
RESULT_DIR = "./results"
for _d in (CKPT_DIR, PLOT_DIR, RESULT_DIR):
    os.makedirs(_d, exist_ok=True)

configs = [{"LOOKBACK": lb, "n1": n1} for lb in LOOKBACK_LIST for n1 in N1_LIST]
print(f"{N_STOCKS} inds, H={HORIZON}, x_max {XMAX_ALL}, config {len(configs)}")

## 2. Data and folds

In [ ]:
inds = pd.read_csv(f"csv/{N_STOCKS}_industry.csv")
inds["Date"] = pd.to_datetime(inds["Date"])
inds = inds.set_index("Date").sort_index()
inds = inds[~inds.index.duplicated(keep="first")] / 100.0

stock_names = inds.columns.tolist()
full_np     = inds.values
full_dates  = inds.index
d, C_cap    = 1.0, 1.0

def date_to_idx(s):
    return full_dates.searchsorted(pd.Timestamp(s), side="left")

def make_folds(horizon):
    fs = []
    for f in range(N_FOLDS):
        ty = 2018 + f
        fs.append({
            "fold"          : f + 1,
            "train_end_idx" : date_to_idx(f"{ty - VAL_YEARS}-01-01"),
            "val_start_idx" : date_to_idx(f"{ty - VAL_YEARS}-01-01"),
            "val_end_idx"   : date_to_idx(f"{ty}-01-01"),
            "test_start_idx": date_to_idx(f"{ty}-01-01"),
            "test_end_idx"  : min(date_to_idx(f"{ty + TEST_YEARS}-01-01") + horizon,
                                  len(full_np)),
            "val_year"      : f"{ty - VAL_YEARS}~{ty - 1}",
            "test_year"     : ty,
        })
    return fs

folds = make_folds(HORIZON)
N_WIN = sum(len(rebalance_dates(f, 252, HORIZON, REBAL)) for f in folds)
print(f"{len(full_np)} days, {full_dates[0]:%Y-%m-%d} to {full_dates[-1]:%Y-%m-%d}")
print(f"{len(folds)} folds, {N_WIN} rebalancing windows per config")

## 3. Training

```
python run_xmax_sweep.py --data 10 --horizon 126 --xmax 0.6 0.3 0.2
```

The sweep runs one x_max at a time: 16 DFL-MDD shards, then the merge, then 4
DFL-MVO shards, before moving on. Because each cap finishes completely, the
analysis below can be run on the earlier caps while the later ones are still
training.


## 4. Load results

Load DFL-MDD (with carry-forward), DFL-MVO and the benchmarks for one x_max at a
time. `x_max=1.0` uses the existing untagged checkpoints.

**The same cap is applied to the benchmarks** via `build_bench_store(..., x_max=xm)`.

### PTO baselines

In [ ]:
import torch
from dfl_mdd import PredictionModel
from pto_mdd import train_pto_mdd, backtest_pto_mdd
from pto_mvo import train_pto_mvo, backtest_pto_mvo

HIDDEN_DIM, EPOCHS, BATCH_SIZE, LR, PATIENCE = 128, 100, 32, 1e-4, 20
gamma, x_min = 0.0, 0.0
is_mean = full_np[:date_to_idx("2013-01-01")].mean(axis=0)
is_std  = full_np[:date_to_idx("2013-01-01")].std(axis=0)

def make_windows(data, lookback, horizon, start, end):
    """Keep float64, as the other notebooks do, so the backtest Sigma path matches."""
    out = []
    for t in range(max(start, lookback), end - horizon + 1):
        z = (data[t - lookback:t] - is_mean) / (is_std + 1e-8)
        out.append((z.flatten(), data[t:t + horizon]))
    return out

#XM_RUN = [1.0] + XMAX_LIST          # same model, compared across caps
XM_RUN = [0.2]

for MODEL in ("pto_mdd", "pto_mvo"):
    is_mdd = MODEL == "pto_mdd"
    keys = ([(c["LOOKBACK"], c["n1"]) for c in configs] if is_mdd
            else list(LOOKBACK_LIST))
    maps = {xm: {k: [] for k in keys} for xm in XM_RUN}

    for fold_info in folds:
        print(f"\n-- {MODEL} fold {fold_info['fold']} (test={fold_info['test_year']}) --")
        torch.manual_seed(123); np.random.seed(123)

        seen = set()
        for cfg in configs:
            LB, n1 = cfg["LOOKBACK"], cfg["n1"]
            key = (LB, n1) if is_mdd else LB
            if not is_mdd:
                if LB in seen:                 # PTO-MVO ignores n1, so once per lookback
                    continue
                seen.add(LB)

            tr = make_windows(full_np, LB, HORIZON, LB, fold_info["train_end_idx"])
            va = make_windows(full_np, LB, HORIZON, fold_info["val_start_idx"],
                              fold_info["val_end_idx"])[::HORIZON]
            rb = make_windows(full_np, LB, HORIZON, fold_info["test_start_idx"],
                              fold_info["test_end_idx"])[::REBAL]
            if len(rb) == 0:
                print(f"   LB={LB}: no rebalancing windows, skipped"); continue

            # -- training does not depend on x_max, so it runs once --
            model = PredictionModel(LB * N_STOCKS, HIDDEN_DIM, HORIZON, N_STOCKS)
            trainer = train_pto_mdd if is_mdd else train_pto_mvo
            trainer(model, tr, va, EPOCHS, BATCH_SIZE, LR, patience=PATIENCE)

            # -- backtest repeated per x_max --
            for xm in XM_RUN:
                if is_mdd:
                    bt = backtest_pto_mdd(
                        model, rb, HORIZON, d, C_cap, n1=n1,
                        x_min=x_min, x_max=xm, gamma=gamma, delta=DELTA,
                        is_mean=is_mean, is_std=is_std,
                        stock_names=stock_names, rebal=REBAL, solve_method=SOLVER)
                else:
                    bt = backtest_pto_mvo(
                        model, rb, HORIZON, d, C_cap, delta=DELTA,
                        x_min=x_min, x_max=xm, gamma=gamma,
                        is_mean=is_mean, is_std=is_std,
                        stock_names=stock_names, rebal=REBAL, solve_method=SOLVER)
                maps[xm][key].extend(bt)
                mx = max(float(np.max(r["weights"])) for r in bt)
                print(f"      x_max={xm:<4g} windows {len(bt)}, max weight {mx:.4f}"
                      + ("  <-- cap violated" if mx > xm + 1e-6 else ""))

    # -- save per x_max --
    for xm in XM_RUN:
        tag = xm_tag(xm)
        out = (f"{CKPT_DIR}/{MODEL}_{N_STOCKS}_inds_h{HORIZON}{tag}"
               f"_d{DELTA}_{SOLVER}.pkl")
        with open(out, "wb") as f:
            pickle.dump({"fold_results_map": maps[xm], "completed_fold": len(folds),
                         "delta_val": DELTA, "horizon": HORIZON,
                         "x_max": xm, "solver": SOLVER}, f)
        n = {len(v) for v in maps[xm].values()}
        print(f"   {os.path.basename(out)}  config {len(maps[xm])}, window {n}")

In [ ]:
def xm_tag(xm):
    return "" if xm >= 1.0 else f"_xm{xm:g}"

def load_xmax(xm, verbose=True):
    """Load every model's results for a single x_max.

    Returns a dict
      cf    : DFL-MDD + carry-forward   {(DELTA, lam): [(results, label), ...]}
      mvo   : DFL-MVO                   {(DELTA, lam): [(results, label), ...]}
      pmdd  : PTO-MDD                   [(results, label), ...]   (independent of lam)
      pmvo  : PTO-MVO                   [(results, label), ...]   (independent of lam)
      bench : benchmarks                {label: results}          (same x_max applied)
    Missing models come back as an empty dict or list so later cells skip them quietly.
    """
    tag = xm_tag(xm)
    miss = []

    # -- DFL-MDD (+ carry-forward) --
    store, infs = {}, {}
    for lam in LAM_LIST:
        p = (f"{CKPT_DIR}/dfl_mdd_{N_STOCKS}_inds_h{HORIZON}{tag}"
             f"_d{DELTA}_l{lam}_{SOLVER}.pkl")
        if not os.path.exists(p):
            miss.append(f"dfl_mdd lam={lam}"); continue
        with open(p, "rb") as f: ck = pickle.load(f)
        store[(DELTA, lam)] = [(ck["fold_results_map"][(c["LOOKBACK"], c["n1"])],
                                f"DFL-MDD (LB={c['LOOKBACK']}, n1={c['n1']})")
                               for c in configs]
        infs[(DELTA, lam)] = ck["infeas_map"]
    if not store:
        if verbose: print(f"  x_max={xm:<4g} no DFL-MDD, skipped")
        return None

    cf = apply_carryforward(store, infs, folds=folds, full_np=full_np,
                            HORIZON=HORIZON, REBAL=REBAL, d=d, C=C_cap,
                            verbose=False)
    cf = {k: [(attach_date_idx(r, folds, parse_lb(l), HORIZON, REBAL), l)
              for r, l in v] for k, v in cf.items()}

    # -- DFL-MVO --
    mvo = {}
    for lam in LAM_LIST:
        p = (f"{CKPT_DIR}/dfl_mvo_{N_STOCKS}_inds_h{HORIZON}{tag}"
             f"_d{DELTA}_l{lam}_{SOLVER}.pkl")
        if not os.path.exists(p):
            miss.append(f"dfl_mvo lam={lam}"); continue
        with open(p, "rb") as f: ck = pickle.load(f)
        mvo[(DELTA, lam)] = [(attach_date_idx(ck["fold_results_map"][lb], folds,
                                              lb, HORIZON, REBAL),
                              f"DFL-MVO (LB={lb})") for lb in LOOKBACK_LIST]

    # -- PTO-MDD (independent of lam, 8 configs) --
    pmdd = []
    p = (f"{CKPT_DIR}/pto_mdd_{N_STOCKS}_inds_h{HORIZON}{tag}"
         f"_d{DELTA}_{SOLVER}.pkl")
    if os.path.exists(p):
        with open(p, "rb") as f: ck = pickle.load(f)
        pmdd = [(attach_date_idx(ck["fold_results_map"][(c["LOOKBACK"], c["n1"])],
                                 folds, c["LOOKBACK"], HORIZON, REBAL),
                 f"PTO-MDD (LB={c['LOOKBACK']}, n1={c['n1']})") for c in configs]
    else:
        miss.append("pto_mdd")

    # -- PTO-MVO (independent of lam, 2 lookbacks) --
    pmvo = []
    p = (f"{CKPT_DIR}/pto_mvo_{N_STOCKS}_inds_h{HORIZON}{tag}"
         f"_d{DELTA}_{SOLVER}.pkl")
    if os.path.exists(p):
        with open(p, "rb") as f: ck = pickle.load(f)
        pmvo = [(attach_date_idx(ck["fold_results_map"][lb], folds,
                                 lb, HORIZON, REBAL),
                 f"PTO-MVO (LB={lb})") for lb in LOOKBACK_LIST
                if lb in ck["fold_results_map"]]
    else:
        miss.append("pto_mvo")

    # -- benchmarks, computed on the fly with the same cap --
    bench, _ = build_bench_store(full_np, folds, stock_names, LOOKBACK_LIST,
                                 HORIZON, REBAL, delta=DELTA, x_max=xm,
                                 verbose=False)

    if verbose:
        def _maxw(items):
            return max((float(np.max(r["weights"])) for res, _ in items
                        for r in res), default=float("nan"))
        parts = [("DFL-MDD", cf[(DELTA, LAM_LIST[0])]),
                 ("DFL-MVO", mvo.get((DELTA, LAM_LIST[0]), [])),
                 ("PTO-MDD", pmdd), ("PTO-MVO", pmvo),
                 ("bench", [(v, k) for k, v in bench.items()])]
        n = len(cf[(DELTA, LAM_LIST[0])][0][0])
        print(f"  x_max={xm:<4g} window {n}")
        for nm, items in parts:
            if not items:
                print(f"     {nm:<9} missing"); continue
            mw = _maxw(items)
            flag = "  <-- cap violated" if mw > xm + 1e-6 else ""
            print(f"     {nm:<9} {len(items)}, max weight {mw:.4f}{flag}")
        if miss:
            print(f"     missing: {miss}")

    return {"cf": cf, "mvo": mvo, "pmdd": pmdd, "pmvo": pmvo, "bench": bench}

DATA = {}
for xm in XMAX_ALL:
    r = load_xmax(xm)
    if r: DATA[xm] = r
print(f"\nloaded x_max: {sorted(DATA)}")

In [ ]:
# ==========================================================
#  DFL-MDD vs DFL-MVO, compared per weight cap
#    DFL-MDD is averaged over the 4 n1 values; DFL-MVO has one run per lookback
#    the gap is the isolated effect of the drawdown constraint, all else equal
# ==========================================================
from scipy import stats

def _mean_metric(res_list, key):
    return float(np.mean([compute_performance(r)[key] for r in res_list]))

rows = []
for xm in sorted(DATA):
    D = DATA[xm]
    for lam in LAM_LIST:
        if (DELTA, lam) not in D["cf"] or (DELTA, lam) not in D["mvo"]:
            continue
        for lb in LOOKBACK_LIST:
            mdd_res = [r for r, l in D["cf"][(DELTA, lam)] if parse_lb(l) == lb]
            mvo_res = [r for r, l in D["mvo"][(DELTA, lam)] if parse_lb(l) == lb]
            if not mdd_res or not mvo_res:
                continue
            mvo_res = mvo_res[0]

            row = {"x_max": xm, "lam": lam, "LB": lb}
            for nm, key in [("Ann.Ret(%)", "Ann.Ret"), ("Sharpe", "Sharpe"),
                            ("MDD(%)", "MDD"), ("Calmar", "Calmar"), ("HHI", "HHI")]:
                sc = 100 if "%" in nm else 1
                a = _mean_metric(mdd_res, key) * sc            # DFL-MDD (n1 mean)
                b = compute_performance(mvo_res)[key] * sc     # DFL-MVO
                row[f"MDD_{nm}"] = round(a, 4 if sc == 1 else 2)
                row[f"MVO_{nm}"] = round(b, 4 if sc == 1 else 2)
                row[f"delta_{nm}"]   = round(a - b, 4 if sc == 1 else 2)

            wm = np.concatenate([[r["weights"] for r in x] for x in mdd_res])
            wv = np.array([r["weights"] for r in mvo_res], dtype=float)
            row["MDD_nActive"] = round(float((wm > 1e-4).sum(1).mean()), 2)
            row["MVO_nActive"] = round(float((wv > 1e-4).sum(1).mean()), 2)
            row["MDD_Turnover"] = round(float(np.mean(
                [compute_turnover(r, full_np, REBAL)["mean"] for r in mdd_res])), 4)
            row["MVO_Turnover"] = round(
                compute_turnover(mvo_res, full_np, REBAL)["mean"], 4)

            # -- one-sided paired t-test on per-window MDD, over the common dates --
            def per_date(res_list):
                rec = {}
                for res in res_list:
                    for r in res:
                        rec.setdefault(r["date_idx"], []).append(r["M_real"] * 100)
                return pd.Series({k: np.mean(v) for k, v in rec.items()})
            a_s, b_s = per_date(mdd_res), per_date([mvo_res])
            idx = sorted(set(a_s.index) & set(b_s.index))
            x, y = a_s.loc[idx].values, b_s.loc[idx].values
            diff = x - y
            t, p2 = stats.ttest_rel(x, y)
            p1 = p2 / 2 if t < 0 else 1 - p2 / 2
            row.update({"n": len(idx),
                        "perwin_MDD": round(x.mean(), 2),
                        "perwin_MVO": round(y.mean(), 2),
                        "perwin_delta": round(diff.mean(), 2),
                        "Cohen d": round(abs(diff.mean() / diff.std(ddof=1)), 3),
                        "t": round(t, 2), "p": round(p1, 4),
                        "significant(0.05)": "+" if p1 < 0.05 else ("-" if p1 > 0.95 else "=")})
            rows.append(row)

df_cmp = pd.DataFrame(rows)
# out = f"{RESULT_DIR}/{N_STOCKS}_inds_h{HORIZON}_mdd_vs_mvo_xmax.csv"
# df_cmp.to_csv(out, index=False, encoding="utf-8-sig")
# print(f"saved: {out}  ({len(df_cmp)} rows)\n")

print("-- DFL-MDD vs DFL-MVO  (lam 4 mean) --")
display(df_cmp.groupby(["x_max", "LB"])[
    ["MDD_Ann.Ret(%)", "MVO_Ann.Ret(%)", "MDD_Sharpe", "MVO_Sharpe",
     "MDD_MDD(%)", "MVO_MDD(%)", "MDD_Calmar", "MVO_Calmar"]].mean().round(3))

print("\n-- concentration and turnover --")
display(df_cmp.groupby(["x_max", "LB"])[
    ["MDD_HHI", "MVO_HHI", "MDD_nActive", "MVO_nActive",
     "MDD_Turnover", "MVO_Turnover"]].mean().round(3))

print("\n-- per-window MDD test: is DFL-MDD below DFL-MVO? --")
display(df_cmp[["x_max", "LB", "lam", "n", "perwin_MDD", "perwin_MVO", "perwin_delta",
                "Cohen d", "t", "p", "significant(0.05)"]]
        .sort_values(["x_max", "LB", "lam"]).reset_index(drop=True))

print("\n-- verdict summary (+ = DFL-MDD drawdown significantly lower) --")
display(df_cmp.pivot_table(index=["LB", "lam"], columns="x_max",
                           values="significant(0.05)", aggfunc="first"))

## 5. Performance across x_max

In [ ]:
rows = []
for xm, D in sorted(DATA.items()):
    for lam in LAM_LIST:
        if (DELTA, lam) not in D["cf"]: continue
        groups = [("DFL-MDD",   D["cf"][(DELTA, lam)]),
                ("DFL-MVO",   D["mvo"].get((DELTA, lam), [])),
                ("PTO-MDD",   D["pmdd"]),                              # added
                ("PTO-MVO",   D["pmvo"]),                              # added
                ("Benchmark", [(v, k) for k, v in D["bench"].items()])]
        for g, items in groups:
            for res, lbl in items:
                p  = compute_performance(res)
                to = compute_turnover(res, full_np, REBAL)["mean"]
                w  = np.array([r["weights"] for r in res], dtype=float)
                rows.append({"x_max": xm, "lam": lam, "group": g, "label": lbl,
                             "Ann.Ret(%)": round(p["Ann.Ret"] * 100, 2),
                             "Sharpe": round(p["Sharpe"], 4),
                             "MDD(%)": round(p["MDD"] * 100, 2),
                             "MDD_abs(%)": round(p["MDD_abs"] * 100, 2),
                             "Calmar": round(p["Calmar"], 4),
                             "HHI": round(p["HHI"], 4),
                             "MaxW": round(float(w.max()), 4),
                             "nActive": round(float((w > 1e-4).sum(1).mean()), 2),
                             "Turnover": round(to, 4)})
df_xm = pd.DataFrame(rows)
out = f"{RESULT_DIR}/{N_STOCKS}_inds_h{HORIZON}_xmax_compare.csv"
df_xm.to_csv(out, index=False, encoding="utf-8-sig")
print(f"saved: {out}  ({len(df_xm)} rows)")

print("\n-- DFL-MDD group mean over the 8 configs --")
display(df_xm[df_xm.group == "DFL-MDD"].groupby(["x_max", "lam"])[
    ["Ann.Ret(%)", "Sharpe", "MDD(%)", "Calmar", "HHI", "nActive", "Turnover"]
].mean().round(3))

### 5-1. Concentration: did the cap actually bind?

In [ ]:
display(df_xm[df_xm.group == "DFL-MDD"].pivot_table(
    index="lam", columns="x_max", values=["HHI", "nActive", "MaxW"]).round(3))

viol = df_xm[df_xm.MaxW > df_xm.x_max + 1e-6]
print(f"cap violations: {len(viol)}" + ("" if len(viol) == 0 else "  <-- needs checking"))
if len(viol):
    display(viol[["x_max", "lam", "group", "label", "MaxW"]])

In [ ]:
# ==========================================================
#  every config ranked by MDD, with the x_max values side by side
#  (the weight-cap version of the ranked_MDD figure)
# ==========================================================
from matplotlib.patches import Patch

METRIC = "MDD(%)"          # "Calmar", "Sharpe" or "Turnover" also work
PAL = {"DFL-MDD": "#0B6E8F", "DFL-MVO": "#8FBFCF",
       "PTO-MDD": "#B23A48", "PTO-MVO": "#E0A0A8",
       "Benchmark": "#9AA5B1"}
XMS = sorted(DATA)
ASC = METRIC not in ("Calmar", "Sharpe", "Ann.Ret(%)")   # lower is better for MDD and Turnover

xmax_axis = df_xm[METRIC].max() * 1.13

for LAM in LAM_LIST:
    sub = df_xm[df_xm.lam == LAM]
    if sub.empty:
        continue
    nrow = int(sub.groupby("x_max").size().max())
    fig, axes = plt.subplots(1, len(XMS),
                             figsize=(6.2 * len(XMS), 0.30 * nrow + 1.6),
                             sharex=True)
    axes = np.atleast_1d(axes)

    for ax, xm in zip(axes, XMS):
        s = (sub[sub.x_max == xm]
             .groupby(["group", "label"])[METRIC].mean()
             .reset_index()
             .sort_values(METRIC, ascending=ASC))
        y = np.arange(len(s))
        ax.barh(y, s[METRIC], height=0.72, edgecolor="none",
                color=[PAL.get(g, "#999") for g in s.group])
        for i, v in enumerate(s[METRIC]):
            ax.text(v + xmax_axis * 0.012, i, f"{v:.1f}",
                    va="center", fontsize=7.5, color="#333")
        ax.set_yticks(y)
        ax.set_yticklabels(s.label, fontsize=7.5)
        ax.invert_yaxis()
        ax.set_xlim(0, xmax_axis)
        ax.set_xlabel(METRIC, fontsize=10)
        ax.set_title(f"$x_{{max}}$ = {xm:g}", fontsize=11.5, pad=8)
        ax.grid(axis="x", alpha=0.22, lw=0.7)
        for sp in ("top", "right", "left"):
            ax.spines[sp].set_visible(False)

    axes[-1].legend(handles=[Patch(facecolor=c, label=g) for g, c in PAL.items()
                             if g in set(sub.group)],
                    fontsize=8, loc="upper right", frameon=False)
    fig.suptitle(f"All configurations ranked by {METRIC}  "
                 f"({N_STOCKS} inds, H={HORIZON}, $\\lambda$={LAM})",
                 fontsize=13, fontweight="bold")
    plt.tight_layout(rect=[0, 0, 1, 0.94])

    m_tag = (METRIC.replace("(%)", "").replace("(", "")
             .replace(")", "").replace("%", ""))
    out = (f"{PLOT_DIR}/ranked_{m_tag}_xmax_{N_STOCKS}_inds"
           f"_h{HORIZON}_lam{LAM}.png")
    plt.savefig(out, bbox_inches="tight", dpi=600)
    plt.show()
    print(f"   {out}")

## 6. Transaction costs

In [ ]:
TCS = [0, 5, 10, 20, 40]
rows = []
for xm, D in sorted(DATA.items()):
    for lam in LAM_LIST:
        if (DELTA, lam) not in D["cf"]: continue
        groups = [("DFL-MDD",   D["cf"][(DELTA, lam)]),
                  ("DFL-MVO",   D["mvo"].get((DELTA, lam), [])),
                  ("PTO-MDD",   D.get("pmdd", [])),
                  ("PTO-MVO",   D.get("pmvo", [])),
                  ("Benchmark", [(v, k) for k, v in D["bench"].items()])]
        for tc in TCS:
            for g, items in groups:
                for res, lbl in items:
                    r2 = apply_tc(res, tc / 1e4, full_np, REBAL) if tc else res
                    p  = compute_performance(r2)
                    to = compute_turnover(res, full_np, REBAL)["mean"]
                    w  = np.array([r["weights"] for r in res], dtype=float)
                    rows.append({"x_max": xm, "lam": lam, "tc_bps": tc,
                                 "group": g, "label": lbl,
                                 "Ann.Ret(%)": round(p["Ann.Ret"] * 100, 2),
                                 "Sharpe": round(p["Sharpe"], 4),
                                 "CVaR(5%)(%)": round(p["CVaR(5%)"] * 100, 2),
                                 "MDD(%)": round(p["MDD"] * 100, 2),
                                 "MDD_abs(%)": round(p["MDD_abs"] * 100, 2),
                                 "Calmar": round(p["Calmar"], 4),
                                 "HHI": round(p["HHI"], 4),
                                 "MaxW": round(float(w.max()), 4),
                                 "Turnover": round(to, 4)})
df_tc = pd.DataFrame(rows)

# -- combined table --
allout = f"{RESULT_DIR}/{N_STOCKS}_inds_h{HORIZON}_xmax_tc_all.csv"
df_tc.to_csv(allout, index=False, encoding="utf-8-sig")
print(f"combined: {os.path.basename(allout)}  ({len(df_tc)} rows)\n")

# -- one file per (x_max, lam) --
for xm in sorted(df_tc.x_max.unique()):
    for lam in sorted(df_tc.lam.unique()):
        s = df_tc[(df_tc.x_max == xm) & (df_tc.lam == lam)]
        if s.empty: continue
        out = (f"{RESULT_DIR}/{N_STOCKS}_inds_h{HORIZON}"
               f"_xm{xm:g}_tc_cf_lam{lam}.csv")
        s.drop(columns=["x_max", "lam"]).to_csv(out, index=False,
                                                encoding="utf-8-sig")
        print(f"   {os.path.basename(out)}  ({len(s)} rows)")

print("\n-- Calmar: tc x x_max (DFL-MDD mean) --")
display(df_tc[df_tc.group == "DFL-MDD"].pivot_table(
    index="tc_bps", columns="x_max", values="Calmar").round(4))
print("-- Turnover: x_max x group --")
display(df_tc[df_tc.tc_bps == 0].pivot_table(
    index="group", columns="x_max", values="Turnover").round(4))

## 7. Cumulative return

In [ ]:
XM_COL = {1.0: "#9AA5B1", 0.6: "#0B6E8F", 0.3: "#B23A48"}

for lam in LAM_LIST:
    avail = [xm for xm in sorted(DATA) if (DELTA, lam) in DATA[xm]["cf"]]
    if not avail: continue
    fig, axes = plt.subplots(1, len(LOOKBACK_LIST), figsize=(13, 4.4), sharey=True)
    axes = np.atleast_1d(axes)
    for ax, lb in zip(axes, LOOKBACK_LIST):
        for xm in avail:
            sel = [res for res, l in DATA[xm]["cf"][(DELTA, lam)] if parse_lb(l) == lb]
            if not sel: continue
            eqs = [build_equity_curve(r) for r in sel]
            L   = min(len(e) for e in eqs)
            eq  = np.mean([e[:L] for e in eqs], axis=0)      # n1 4 mean
            mdd = np.mean([compute_performance(r)["MDD"] for r in sel])
            cal = np.mean([compute_performance(r)["Calmar"] for r in sel])
            ax.plot(eq, color=XM_COL.get(xm, "#666"), lw=1.7,
                    label=f"$x_{{max}}$={xm:g}   MDD {mdd:.1%}  Cal {cal:.2f}")
        ax.axhline(1.0, color="gray", ls="--", lw=0.8, alpha=0.6)
        ax.set_title(f"Lookback = {lb}", fontsize=11.5)
        ax.set_xlabel("Trading days (test period)")
        ax.legend(fontsize=8, loc="upper left")
        ax.grid(alpha=0.22, lw=0.7)
        for sp in ("top", "right"): ax.spines[sp].set_visible(False)
    axes[0].set_ylabel("Portfolio value ($n_1$ mean)")
    fig.suptitle(f"Weight cap comparison  (H={HORIZON}, $\\lambda$={lam})",
                 fontsize=13, fontweight="bold")
    plt.tight_layout(rect=[0, 0, 1, 0.93])
    out = f"{PLOT_DIR}/cumret_xmax_{N_STOCKS}_inds_h{HORIZON}_lam{lam}.png"
    plt.savefig(out, bbox_inches="tight", dpi=300)
    plt.show()
    print(f"   {out}")

## 8. Statistical tests

The same one-sided paired t-test as elsewhere, run per x_max. DFL-MDD, DFL-MVO and
the benchmarks all share the cap, so the comparison stays fair.

In [ ]:
from scipy import stats

ALPHAS = [0.10, 0.05, 0.01]

def sig(p1, a):
    if not np.isfinite(p1): return "n/a"
    return "" if p1 < a else ("" if p1 > 1 - a else "-")

def ttest_for(xm):
    """Per-window MDD test of DFL-MDD against every other model, for one x_max."""
    D = DATA[xm]

    def dates_of(st):
        return {r["date_idx"] for res, _ in st for r in res}

    # -- common dates: the intersection across every model in the comparison --
    sets = {"DFL-MDD": dates_of(D["cf"][(DELTA, LAM_LIST[0])]),
            "DFL-MVO": dates_of(D["mvo"][(DELTA, LAM_LIST[0])])}
    if D.get("pmdd"): sets["PTO-MDD"] = dates_of(D["pmdd"])
    if D.get("pmvo"): sets["PTO-MVO"] = dates_of(D["pmvo"])
    for l, r in D["bench"].items():
        sets[l] = {x["date_idx"] for x in r}
    COMMON = sorted(set.intersection(*sets.values()))

    def per_date(st):
        rec = {}
        for res, _ in st:
            for r in res:
                rec.setdefault(r["date_idx"], []).append(r["M_real"] * 100)
        return pd.Series({k: np.mean(v) for k, v in rec.items()}).loc[COMMON]

    def by_lb(st, lb):
        return [(r, l) for r, l in st if parse_lb(l) == lb]

    out = []
    for lb in LOOKBACK_LIST:
        for lam in LAM_LIST:
            base = per_date(by_lb(D["cf"][(DELTA, lam)], lb))

            comps = {"DFL-MVO":  by_lb(D["mvo"][(DELTA, lam)], lb),
                     "PTO-MDD":  by_lb(D.get("pmdd", []), lb),   # independent of lam
                     "PTO-MVO":  by_lb(D.get("pmvo", []), lb),   # independent of lam
                     "GMV":      [(D["bench"][f"GMV (LB={lb})"], "")],
                     "hist-MVO": [(D["bench"][f"hist-MVO (LB={lb})"], "")],
                     "EW":       [(D["bench"]["EW"], "")]}

            for nm, st in comps.items():
                if not st:                      # model not loaded, skip
                    continue
                x, y = base.values, per_date(st).values
                diff = x - y
                t, p2 = stats.ttest_rel(x, y)
                p1 = p2 / 2 if t < 0 else 1 - p2 / 2
                try:
                    _, w2 = stats.wilcoxon(x, y)
                    w1 = w2 / 2 if np.median(diff) < 0 else 1 - w2 / 2
                except ValueError:
                    w1 = np.nan
                out.append({"x_max": xm, "LB": lb, "lam": lam, "comparison": nm,
                            "n": len(COMMON),
                            "DFL-MDD": round(x.mean(), 2),
                            "other": round(y.mean(), 2),
                            "difference": round(diff.mean(), 2),
                            "Cohen d": round(abs(diff.mean() / diff.std(ddof=1)), 3),
                            "t": round(t, 2), "p (one-sided)": round(p1, 4),
                            **{f"significant({a:.2f})": sig(p1, a) for a in ALPHAS},
                            "Wilcoxon p": round(w1, 4)})
    return pd.DataFrame(out)

res_all = pd.concat([ttest_for(xm) for xm in sorted(DATA)], ignore_index=True)
out = f"{RESULT_DIR}/{N_STOCKS}_inds_h{HORIZON}_xmax_ttest.csv"
res_all.to_csv(out, index=False, encoding="utf-8-sig")
print(f"saved: {out}  ({len(res_all)} rows)")
print(f"  x_max {sorted(res_all.x_max.unique())}, "
      f"comparisons {sorted(res_all['comparison'].unique())}\n")

for xm in sorted(DATA):
    print(f"[x_max = {xm:g}]  alpha = 0.05")
    display(res_all[res_all.x_max == xm].pivot_table(
        index="comparison", columns=["LB", "lam"],
        values="significant(0.05)", aggfunc="first"))